> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board and uploads the
> package. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host,
>   writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

# Hardware and configuration

| | Host run | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | none: `show_reg` is checked against a stub device | none: the same stub device |
| **Result** | not run yet on the hardware | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): `debug.py` moved from hand-written to this notebook, with its first
tests. The API, the output format and the tally are unchanged: every notebook's hardware
section uses `check`, `check_raises`, `check_reset`, `check_summary` and `check_counts`, and
the host reads `check_counts()` with `%eval`.

# Python file
location: `src/micropython_i2c/debug.py`

tests: `tests/test_debug.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/debug.py
"""Bring-up helpers for notebooks: register display and PASS/FAIL checks.

``check`` keeps a running tally so a notebook section can end with ``check_summary()``.

Generated by ``dev_notebooks/debug_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None: the helpers only format and print.

## Internal
None. `show_reg` takes any object with `REG_BYTES` and `reg_read(reg, nbytes)` (a
`RegisterI2CBase` driver), so it needs no import.

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_debug.py
import pytest

from micropython_i2c.debug import (
    bin_bytes,
    check,
    check_counts,
    check_raises,
    check_reset,
    check_summary,
    info,
    show,
    show_reg,
)

Notebook-only: the imports the validation and test cells use. The helpers themselves come
from the code cells below.

In [ ]:
import pytest

# Intent
**Purpose:** the small printing helpers a bring-up notebook needs on the board: show a
register as hex, grouped binary and decimal; and print `PASS`/`FAIL` lines that add up to a
tally at the end of a section.

**Why it exists:** every hardware section ends the same way: `check_summary()` on the board,
`check_counts()` read by the host with `%eval`, the board released, and the run failed on the
host if anything failed. One module keeps the line format and the tally the same in every
notebook.

```mermaid
flowchart LR
    R[check_reset] --> C[check / check_raises]
    C -->|PASS/FAIL line, tally +1| C
    C --> S[check_summary: print the tally]
    S --> H["%%host: %eval check_counts()"]
    H --> D[%board disconnect]
    D --> F[fail the run if failed > 0]
```

**The tally** is one module-level list, `[passed, failed]`, shared by every caller since the
last `check_reset()`. `info` lines are not counted.

**Line formats** (these are what the tests pin down):

| Call | Prints |
|---|---|
| `check("x", True)` | `  [PASS] x` padded to 36 characters, a space, the detail |
| `check_raises(...)` when it raises | the `PASS` line, then 10 spaces and the exception message |
| `check_raises(...)` when it does not | `  [FAIL] <label> ... it did not raise` |
| `info("m")` | `  [info] m` |
| `check_summary()` | `<passed> passed, <failed> failed` |
| `show(0x60A0, "config")` | `config           0x60a0  01100000 10100000  24736` |

**Not its job:** logging or anything the drivers call. Only notebooks use it.

## What the module defines

| Name | What it is |
|---|---|
| `bin_bytes` | an int as space-separated 8-bit groups, MSB first |
| `show` | print a value as hex, grouped binary and decimal |
| `show_reg` | read a register (raw) and `show` it |
| `check` | print a `PASS`/`FAIL` line and count it |
| `check_raises` | a `check` that a call raises a given exception |
| `info` | print an uncounted information line |
| `check_reset` | zero the tally |
| `check_counts` | the tally, without printing |
| `check_summary` | print and return the tally |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
__all__ = [
    "bin_bytes",
    "check",
    "check_counts",
    "check_raises",
    "check_reset",
    "check_summary",
    "info",
    "show",
    "show_reg",
]

## Register display
Three helpers for looking at register values while bringing a part up. The width is always
explicit (`nbytes`), because a width inferred from the value hides a zero high byte: `0x00A0`
and `0xA0` are different registers.

### `bin_bytes`
An int as 8-bit binary groups, most significant byte first, so a bitfield can be read off
against the datasheet's bit numbering.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def bin_bytes(value, nbytes=2):
    """Format an int as space-separated 8-bit groups, MSB first."""
    return " ".join("{:08b}".format((value >> (8 * i)) & 0xFF) for i in reversed(range(nbytes)))

### `show`
One line per value: label, hex padded to the width, the binary groups, and decimal. A value
that does not fit the width raises `ValueError` rather than printing a misleading line.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def show(value, label="", nbytes=2):
    """Print a value as hex, grouped binary and decimal, padded to ``nbytes``.

    The width is explicit because inferring it from magnitude hides a zero high byte.

    Raises:
        ValueError: If ``value`` does not fit in ``nbytes``.
    """
    if not 0 <= value < (1 << (8 * nbytes)):
        raise ValueError("{:#x} does not fit in {} byte(s)".format(value, nbytes))
    print(
        "{:<16} {:#0{}x}  {}  {}".format(
            label, value, nbytes * 2 + 2, bin_bytes(value, nbytes), value
        )
    )

### `show_reg`
Read a register straight from the part (`reg_read`, raw and uncached) and `show` it. The
width defaults to the driver's `REG_BYTES`, the label to the register address.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def show_reg(dev, reg, label="", nbytes=None):
    """Read register ``reg`` from ``dev`` (raw, uncached) and ``show`` it."""
    nbytes = dev.REG_BYTES if nbytes is None else nbytes
    show(dev.reg_read(reg, nbytes), label or "reg {:#04x}".format(reg), nbytes)

### validation/test
A stub register device for `show_reg`, shared with the test file: it answers `reg_read`
from a dictionary and records each call, so a test can see which register and width were
asked for. No bus is involved.

In [ ]:
%%extract ../tests/test_debug.py -a
class StubRegisterDevice:
    """Just enough of a register driver for ``show_reg``: ``REG_BYTES`` and ``reg_read``."""

    REG_BYTES = 2

    def __init__(self, regs):
        self.regs = regs
        self.reads = []

    def reg_read(self, reg, nbytes):
        self.reads.append((reg, nbytes))
        return self.regs[reg]

#### validation: nominal usage
The TMP102's power-on configuration word, a one-byte status, and the same word through a
stub device with the default label.

In [ ]:
print("bin_bytes(0x60A0):", bin_bytes(0x60A0))
show(0x60A0, "config")
show(0x15, "DEVICE_STATUS", 1)
show(0x00A0, "high byte zero")
dev = StubRegisterDevice({0x01: 0x60A0})
show_reg(dev, 0x01)
print("reads:", dev.reads)

#### validation: anti-nominal usage
A value wider than `nbytes` is refused.

In [ ]:
try:
    show(0x1FF, "status", 1)  # 0x1FF needs two bytes
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("show(0x1FF, nbytes=1) should have raised ValueError")

A negative value is refused too: registers are unsigned.

In [ ]:
try:
    show(-1, "temp")  # sign-extended readings are not register values
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("show(-1) should have raised ValueError")

#### unit

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_bin_bytes_groups_msb_first():
    assert bin_bytes(0x60A0) == "01100000 10100000"
    assert bin_bytes(0x15, 1) == "00010101"
    assert bin_bytes(0x0102, 3) == "00000000 00000001 00000010"

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_show_line_format(capsys):
    show(0x60A0, "config")
    show(0x15, "DEVICE_STATUS", 1)
    assert capsys.readouterr().out == (
        "config           0x60a0  01100000 10100000  24736\n"
        "DEVICE_STATUS    0x15  00010101  21\n"
    )

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_show_pads_a_zero_high_byte(capsys):
    show(0x00A0)
    assert capsys.readouterr().out.split()[0] == "0x00a0"

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_show_refuses_a_value_that_does_not_fit(capsys):
    with pytest.raises(ValueError, match="0x1ff does not fit in 1 byte"):
        show(0x1FF, nbytes=1)
    with pytest.raises(ValueError, match="does not fit in 2 byte"):
        show(-1)
    assert capsys.readouterr().out == ""

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_show_reg_uses_reg_bytes_and_the_address_label(capsys):
    dev = StubRegisterDevice({0x01: 0x60A0})
    show_reg(dev, 0x01)
    assert dev.reads == [(0x01, 2)]
    assert capsys.readouterr().out.startswith("reg 0x01         0x60a0  ")

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_show_reg_width_and_label_override(capsys):
    dev = StubRegisterDevice({0x0D: 0x0F})
    show_reg(dev, 0x0D, "CONV_STATUS", 1)
    assert dev.reads == [(0x0D, 1)]
    assert capsys.readouterr().out == "CONV_STATUS      0x0f  00001111  15\n"

## Checks and the tally
`check` prints one line and counts it; everything else in this section builds on it or
reads the count. The tally is a module-level list rather than a class, so a notebook calls
plain functions, and the host can read it with one `%eval check_counts()`.

### The tally
`[passed, failed]` since the last `check_reset()`. A list so the functions can change it
without `global`.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
_tally = [0, 0]  # passed, failed

### `check`
Print a `PASS` or `FAIL` line, count it, and return the result as a `bool`, so a cell can
also branch on it. `passed` may be any value; its truth is what counts.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def check(label, passed, detail=""):
    """Print a PASS/FAIL line, add it to the tally, and return ``passed``."""
    passed = bool(passed)
    _tally[0 if passed else 1] += 1
    print("  [{}] {:<36} {}".format("PASS" if passed else "FAIL", label, detail))
    return passed

### `check_raises`
A `check` that a call raises `exc_type`. On a pass it prints the exception's message under
the line and returns the exception, so a cell can check its attributes. An exception of
another type is not caught: it is a bug in the check, not a failed check.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def check_raises(label, exc_type, fn, *args, **kwargs):
    """Check that ``fn(*args, **kwargs)`` raises ``exc_type``. Returns the exception or None."""
    try:
        fn(*args, **kwargs)
    except exc_type as e:
        check(label, True)
        print("         ", e)
        return e
    check(label, False, "it did not raise")
    return None

### `info`
A line that reads like a check but is not counted: a measured value, a skipped step.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def info(message):
    """Print an informational line (not counted as pass or fail)."""
    print("  [info] " + message)

### `check_reset`
Zero the tally. A hardware section calls it once, right after the reload cell.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def check_reset():
    """Zero the tally."""
    _tally[0] = _tally[1] = 0

### `check_counts`
The tally as a tuple, without printing: the value the host reads with `%eval`, where printed
output would get in the way.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def check_counts():
    """``(passed, failed)`` since the last ``check_reset()``, without printing.

    For reading the tally from the host: ``get_ipython().run_line_magic("eval",
    "check_counts()")`` in a ``%%host`` cell.
    """
    return _tally[0], _tally[1]

### `check_summary`
Print the tally and return it. The last board cell of every hardware section.

In [ ]:
%%extract ../src/micropython_i2c/debug.py -a
def check_summary():
    """Print and return ``(passed, failed)`` since the last ``check_reset()``."""
    passed, failed = _tally
    print("{} passed, {} failed".format(passed, failed))
    return passed, failed

### validation/test

#### validation: nominal usage
A short section the way a hardware section writes it: reset, a few checks (one failing on
purpose), an `info` line, a `check_raises`, the summary. Then reset, so the tally is left as
it was found.

In [ ]:
check_reset()
check("WHO_AM_I", 0x35 == 0x35, "0x35")
check("conversion ready", False, "timed out (on purpose)")
info("temperature 22.9 C")
err = check_raises("bad channel refused", ValueError, int, "not a number")
print("returned:", type(err).__name__, "| counts:", check_counts())
check_summary()
check_reset()

#### validation: anti-nominal usage
`check_raises` reports a call that does not raise as a `FAIL` (it does not raise itself).

In [ ]:
check_reset()
print("returned:", check_raises("int('7') raises", ValueError, int, "7"), "| counts:", check_counts())
check_reset()

An exception of another type goes straight through `check_raises`, uncounted: the check
itself is wrong.

In [ ]:
check_reset()
try:
    check_raises("wrong expectation", KeyError, int, "x")  # int("x") raises ValueError
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("check_raises should have let the ValueError through")
finally:
    print("counts:", check_counts())
    check_reset()

#### unit
Each test starts from `check_reset()`: the tally is shared module state.

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_counts_passes_and_fails_and_returns_the_result():
    check_reset()
    assert check("a", True) is True
    assert check("b", False) is False
    assert check("c", 1) is True
    assert check_counts() == (2, 1)
    check_reset()

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_coerces_the_result_to_bool():
    check_reset()
    assert check("non-empty list", [0]) is True
    assert check("zero", 0) is False
    assert check("None", None) is False
    assert check_counts() == (1, 2)
    check_reset()

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_line_format(capsys):
    check_reset()
    check("WHO_AM_I", True, "0x35")
    check("ready", False)
    assert capsys.readouterr().out == (
        "  [PASS] WHO_AM_I" + " " * 28 + " 0x35\n"
        "  [FAIL] ready" + " " * 31 + " \n"
    )
    check_reset()

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_line_keeps_a_long_label_whole(capsys):
    check_reset()
    label = "x" * 40
    check(label, True, "d")
    assert capsys.readouterr().out == "  [PASS] " + label + " d\n"
    check_reset()

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_raises_passes_and_returns_the_exception(capsys):
    check_reset()
    err = check_raises("bad int", ValueError, int, "x")
    assert isinstance(err, ValueError)
    assert check_counts() == (1, 0)
    out = capsys.readouterr().out
    assert out == "  [PASS] bad int" + " " * 29 + " \n" + " " * 10 + str(err) + "\n"
    check_reset()

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_raises_fails_when_nothing_is_raised(capsys):
    check_reset()
    assert check_raises("int('7')", ValueError, int, "7") is None
    assert check_counts() == (0, 1)
    assert capsys.readouterr().out == "  [FAIL] int('7')" + " " * 28 + " it did not raise\n"
    check_reset()

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_raises_lets_another_exception_through():
    check_reset()
    with pytest.raises(ValueError):
        check_raises("wrong type", KeyError, int, "x")
    assert check_counts() == (0, 0)

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_raises_passes_args_and_kwargs():
    check_reset()
    seen = []

    def refuse(*args, **kwargs):
        seen.append((args, kwargs))
        raise OSError(19, "ENODEV")

    err = check_raises("refused", OSError, refuse, 1, 2, addr=0x48)
    assert seen == [((1, 2), {"addr": 0x48})]
    assert err.args == (19, "ENODEV")
    check_reset()

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_info_is_printed_but_not_counted(capsys):
    check_reset()
    info("temperature 22.9 C")
    assert capsys.readouterr().out == "  [info] temperature 22.9 C\n"
    assert check_counts() == (0, 0)

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_reset_zeroes_the_tally():
    check("a", True)
    check("b", False)
    check_reset()
    assert check_counts() == (0, 0)

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_counts_prints_nothing(capsys):
    check_reset()
    check("a", True)
    capsys.readouterr()
    assert check_counts() == (1, 0)
    assert capsys.readouterr().out == ""
    check_reset()

In [ ]:
%%extract ../tests/test_debug.py -a --strip-ipytest
%%ipytest


def test_check_summary_prints_and_returns_the_tally(capsys):
    check_reset()
    check("a", True)
    check("b", True)
    check("c", False)
    capsys.readouterr()
    assert check_summary() == (2, 1)
    assert capsys.readouterr().out == "2 passed, 1 failed\n"
    assert check_counts() == (2, 1)  # the summary does not reset
    check_reset()

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3 and uploads the
> package. Nothing here uses the I2C bus.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

Layout: connect and set up, the **matching tests** (the helpers' output and tally on
MicroPython, compared with the same strings as the unit tests above), then the tally and
disconnect.

**How a check of `check` works here.** The module under test is also the one that keeps this
section's tally, so each exercise runs in a sandbox: it saves the tally, captures what the
helpers print (by giving `micropython_i2c.debug` its own `print` for the moment), runs, then
restores the tally and the real `print`. The deliberate `FAIL` lines go into the capture,
not onto the screen, and only the checks of the result are counted.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally. The module itself is imported as `dbg` too:
the sandbox below needs it.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

import micropython_i2c.debug as dbg
from micropython_i2c.debug import (
    bin_bytes,
    check,
    check_counts,
    check_raises,
    check_reset,
    check_summary,
    info,
    show,
    show_reg,
)

check_reset()

Notebook-only: the sandbox and the stub device. `sandbox(fn)` runs `fn` with a fresh tally
and `print` captured inside `micropython_i2c.debug`, and returns `(fn's result, the captured
text, the tally fn left)`. It restores the section's tally and `print` even if `fn` raises.

**What to do:** nothing.

In [ ]:
captured = []


def capture(*args, sep=" ", end="\n"):
    captured.append(sep.join([str(a) for a in args]) + end)


def sandbox(fn):
    saved = check_counts()
    check_reset()
    del captured[:]
    dbg.print = capture  # module globals come before builtins
    try:
        result = fn()
        counts = check_counts()
    finally:
        del dbg.print
        dbg._tally[0], dbg._tally[1] = saved
    return result, "".join(captured), counts


class StubRegisterDevice:
    REG_BYTES = 2

    def __init__(self, regs):
        self.regs = regs
        self.reads = []

    def reg_read(self, reg, nbytes):
        self.reads.append((reg, nbytes))
        return self.regs[reg]

## Matching tests

**Matches** `test_check_counts_passes_and_fails_and_returns_the_result`,
`test_check_coerces_the_result_to_bool` and `test_check_line_format`.

**What to do:** nothing.

In [ ]:
result, out, counts = sandbox(lambda: (check("WHO_AM_I", True, "0x35"), check("ready", False),
                                       check("non-empty", [0]), check("zero", 0)))  # fmt: skip
check("test_check_counts_passes_and_fails", result == (True, False, True, False), str(result))
check("check tally inside the sandbox", counts == (2, 2), str(counts))
check("test_check_line_format",
      out.startswith("  [PASS] WHO_AM_I" + " " * 28 + " 0x35\n"
                     "  [FAIL] ready" + " " * 31 + " \n"))  # fmt: skip
label = "x" * 40
result, out, counts = sandbox(lambda: check(label, True, "d"))
check("test_check_line_keeps_a_long_label_whole", out == "  [PASS] " + label + " d\n", repr(out))

**Matches** `test_check_raises_passes_and_returns_the_exception`,
`test_check_raises_fails_when_nothing_is_raised`,
`test_check_raises_lets_another_exception_through` and
`test_check_raises_passes_args_and_kwargs`.

**What to do:** nothing.

In [ ]:
err, out, counts = sandbox(lambda: check_raises("bad int", ValueError, int, "x"))
check("test_check_raises_passes: returns the exception", isinstance(err, ValueError))
check("test_check_raises_passes: counted", counts == (1, 0), str(counts))
check("test_check_raises_passes: output",
      out == "  [PASS] bad int" + " " * 29 + " \n" + " " * 10 + str(err) + "\n", repr(out))  # fmt: skip
result, out, counts = sandbox(lambda: check_raises("int('7')", ValueError, int, "7"))
check("test_check_raises_fails_when_nothing_is_raised",
      result is None and counts == (0, 1)
      and out == "  [FAIL] int('7')" + " " * 28 + " it did not raise\n", repr(out))  # fmt: skip


def wrong_type():
    try:
        check_raises("wrong type", KeyError, int, "x")
    except ValueError:
        return True
    return False


result, out, counts = sandbox(wrong_type)
check("test_check_raises_lets_another_exception_through", result and counts == (0, 0))
seen = []


def refuse(*args, **kwargs):
    seen.append((args, kwargs))
    raise OSError(19, "ENODEV")


err, out, counts = sandbox(lambda: check_raises("refused", OSError, refuse, 1, 2, addr=0x48))
check("test_check_raises_passes_args_and_kwargs",
      seen == [((1, 2), {"addr": 0x48})] and err.args == (19, "ENODEV"), str(seen))  # fmt: skip

**Matches** `test_info_is_printed_but_not_counted`, `test_check_reset_zeroes_the_tally`,
`test_check_counts_prints_nothing` and `test_check_summary_prints_and_returns_the_tally`.

**What to do:** nothing.

In [ ]:
result, out, counts = sandbox(lambda: info("temperature 22.9 C"))
check("test_info_is_printed_but_not_counted",
      out == "  [info] temperature 22.9 C\n" and counts == (0, 0), repr(out))  # fmt: skip


def reset_after_checks():
    check("a", True)
    check("b", False)
    check_reset()
    return check_counts()


result, out, counts = sandbox(reset_after_checks)
check("test_check_reset_zeroes_the_tally", result == (0, 0))
result, out, counts = sandbox(check_counts)
check("test_check_counts_prints_nothing", result == (0, 0) and out == "", repr(out))


def summary_of_three():
    check("a", True)
    check("b", True)
    check("c", False)
    del captured[:]  # keep only what check_summary prints
    return check_summary()


result, out, counts = sandbox(summary_of_three)
check("test_check_summary_prints_and_returns_the_tally",
      result == (2, 1) and counts == (2, 1) and out == "2 passed, 1 failed\n", repr(out))  # fmt: skip

**Matches** `test_bin_bytes_groups_msb_first`, `test_show_line_format`,
`test_show_pads_a_zero_high_byte` and `test_show_refuses_a_value_that_does_not_fit`:
MicroPython's `str.format`, including the nested width in `{:#0{}x}`.

**What to do:** nothing.

In [ ]:
check("test_bin_bytes_groups_msb_first",
      bin_bytes(0x60A0) == "01100000 10100000" and bin_bytes(0x15, 1) == "00010101"
      and bin_bytes(0x0102, 3) == "00000000 00000001 00000010")  # fmt: skip
result, out, counts = sandbox(lambda: (show(0x60A0, "config"), show(0x15, "DEVICE_STATUS", 1)))
check("test_show_line_format",
      out == "config           0x60a0  01100000 10100000  24736\n"
             "DEVICE_STATUS    0x15  00010101  21\n", repr(out))  # fmt: skip
result, out, counts = sandbox(lambda: show(0x00A0))
check("test_show_pads_a_zero_high_byte", out.split()[0] == "0x00a0", repr(out))
check_raises("test_show_refuses_a_value_that_does_not_fit: too wide", ValueError, show, 0x1FF, "", 1)
check_raises("test_show_refuses_a_value_that_does_not_fit: negative", ValueError, show, -1)

**Matches** `test_show_reg_uses_reg_bytes_and_the_address_label` and
`test_show_reg_width_and_label_override`.

**What to do:** nothing.

In [ ]:
dev = StubRegisterDevice({0x01: 0x60A0, 0x0D: 0x0F})
result, out, counts = sandbox(lambda: show_reg(dev, 0x01))
check("test_show_reg_uses_reg_bytes_and_the_address_label",
      dev.reads == [(0x01, 2)] and out.startswith("reg 0x01         0x60a0  "), repr(out))  # fmt: skip
result, out, counts = sandbox(lambda: show_reg(dev, 0x0D, "CONV_STATUS", 1))
check("test_show_reg_width_and_label_override",
      dev.reads[-1] == (0x0D, 1) and out == "CONV_STATUS      0x0f  00001111  15\n", repr(out))  # fmt: skip

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))